# Ganzer Film in 8K – Direktstream zu Google Drive

**Ablauf:** Google Drive → Colab T4 (7680×4320 HEVC) → sicherer temporärer HTTPS-Tunnel → GitHub Actions (`rclone rcat`) → Google Drive. Keine 8K-Zwischendatei in Colab, kein Video in GitHub-Releases oder -Artefakten. **Ausgabe:** MKV mit HEVC; die erste Original-Tonspur bleibt bitgenau erhalten (auch vorhandenes 5.1). Vorher in Colab den Secret **GH_TOKEN** verfügbar machen (bereits aus 4K eingerichtet).

**Wichtig:** Dies ist eine experimentelle Langzeit-Streaming-Verbindung. Colab-, Cloudflare- oder GitHub-Timeouts können einen langen Film abbrechen. Ein abgebrochener Stream ist **nicht fortsetzbar** und muss neu begonnen werden. Die Notebook-Zelle muss während der Übertragung weiterlaufen. Die Auflösung 8K ist Upscaling, keine echten zusätzlichen Bilddetails.


In [ ]:
# 8K-Ganzfilm: Google Drive -> Colab (GPU) -> GitHub Actions -> Google Drive
# Keine 8K-Ausgabedatei auf Colab und keine Video-Assets/Artefakte auf GitHub.
# In Colab: Secret GH_TOKEN mit Contents: Read/Write fuer dieses Repository.
from google.colab import auth, userdata
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import Request as GoogleAuthRequest
import requests, subprocess, threading, secrets, hmac, time, json, re, os
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlsplit

SOURCE_ID = "1_fD9iNP8uDkV81L1ne-03sTEa6ERj1Ol"
REPO = "nikl223-ctrl/LibreMediaConverter"
OUTPUT_NAME = "Zoomania_8K_HEVC_" + time.strftime("%Y%m%d_%H%M%S") + ".mkv"
GPU_WIDTH, GPU_HEIGHT = 7680, 4320
SOURCE_PORT, VIDEO_PORT = 18766, 18765
auth_lock = threading.Lock()
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])

def access_token():
    with auth_lock:
        if not creds.valid:
            creds.refresh(GoogleAuthRequest())
        return creds.token

gh = userdata.get("GH_TOKEN")
if not gh:
    raise RuntimeError("Colab-Secret GH_TOKEN fehlt. Bitte nicht im Notebook eintippen.")
headers_gh = {
    "Authorization": "Bearer " + gh,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
source_api = "https://www.googleapis.com/drive/v3/files/" + SOURCE_ID
info = requests.get(
    source_api, params={"fields": "name,size"},
    headers={"Authorization": "Bearer " + access_token()}, timeout=30
)
info.raise_for_status()
print("Quelle:", info.json().get("name"), "|", round(int(info.json().get("size",0))/1024**3, 2), "GiB")
print("Ausgabe:", OUTPUT_NAME, "| 7680 x 4320 | HEVC NVENC | erste Tonspur unveraendert")

# T4: HEVC-NVENC kann 8K; H.264-NVENC ist hier nicht geeignet.
subprocess.run(["nvidia-smi"], check=True)
encoders = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True, check=True
).stdout
filters = subprocess.run(
    ["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True, check=True
).stdout
if "hevc_nvenc" not in encoders or "scale_cuda" not in filters:
    raise RuntimeError("FFmpeg benoetigt hevc_nvenc und scale_cuda. Colab GPU-Laufzeit aktivieren.")
print("Pruefe 8K-HEVC-Encoder ohne Videodatei...")
test = subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error",
    "-f", "lavfi", "-i", "color=c=black:s=7680x4320:r=1",
    "-frames:v", "1", "-c:v", "hevc_nvenc", "-f", "null", "-"
], capture_output=True, text=True)
if test.returncode:
    raise RuntimeError("8K-NVENC-Test fehlgeschlagen: " + test.stderr[-1800:])
print("8K-Encoder bereit.")

# Lokaler Quell-Proxy: erneuert Drive-OAuth vor jedem HTTP-Range-Request.
# Nur auf Loopback-Port SOURCE_PORT, der NICHT in den Tunnel geht.
class DriveInput(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, *args):
        pass
    def do_GET(self):
        if urlsplit(self.path).path != "/source":
            self.send_error(404)
            return
        req_headers = {
            "Authorization": "Bearer " + access_token(),
            "Accept-Encoding": "identity",
        }
        if self.headers.get("Range"):
            req_headers["Range"] = self.headers["Range"]
        try:
            with requests.get(source_api, params={"alt": "media"},
                              headers=req_headers, stream=True, timeout=(30, 120)) as upstream:
                self.send_response(upstream.status_code)
                for k in ("Content-Length", "Content-Range", "Accept-Ranges", "Content-Type"):
                    if upstream.headers.get(k):
                        self.send_header(k, upstream.headers[k])
                self.send_header("Connection", "close")
                self.end_headers()
                for block in upstream.iter_content(chunk_size=1024 * 1024):
                    if block:
                        self.wfile.write(block)
        except (BrokenPipeError, ConnectionResetError):
            pass
        except Exception as exc:
            print("Drive-Quellstream unterbrochen:", str(exc)[:300], flush=True)
        finally:
            self.close_connection = True

input_server = ThreadingHTTPServer(("127.0.0.1", SOURCE_PORT), DriveInput)
threading.Thread(target=input_server.serve_forever, daemon=True).start()

# HEVC 8K direkt auf STDOUT als streambares MKV, nicht als Datei.
ffmpeg_cmd = [
    "ffmpeg", "-hide_banner", "-nostdin", "-loglevel", "error",
    "-stats_period", "20", "-progress", "pipe:2",
    "-init_hw_device", "cuda=cuda0:0",
    "-i", "http://127.0.0.1:" + str(SOURCE_PORT) + "/source",
    "-map", "0:v:0", "-map", "0:a:0?",
    "-filter_hw_device", "cuda0",
    "-vf", "format=nv12,hwupload_cuda,scale_cuda=w=7680:h=4320:format=nv12",
    "-c:v", "hevc_nvenc", "-profile:v", "main", "-preset", "p4",
    "-rc", "vbr", "-b:v", "45M", "-maxrate", "60M", "-bufsize", "120M",
    "-c:a", "copy",
    "-f", "matroska", "pipe:1"
]
stream_key = secrets.token_urlsafe(40)
event_started = threading.Event()
encode_finished = threading.Event()
upload_confirmed = threading.Event()
state = {"encoded": False, "uploaded": False, "upload_error": None}
state_lock = threading.Lock()

def progress_reader(pipe):
    last = {}
    for raw in iter(pipe.readline, b""):
        line = raw.decode("utf-8", "replace").strip()
        k, sep, value = line.partition("=")
        if sep:
            last[k] = value
            if k == "progress":
                print("8K: Filmzeit", last.get("out_time", "?"),
                      "| FPS", last.get("fps", "?"),
                      "| Tempo", last.get("speed", "?"), flush=True)
        elif line:
            print("FFmpeg:", line[:350], flush=True)

class VideoOutput(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, *args):
        pass
    def authorized(self):
        return hmac.compare_digest(self.headers.get("X-Stream-Key", ""), stream_key)
    def do_POST(self):
        if urlsplit(self.path).path != "/report" or not self.authorized():
            self.send_error(403)
            return
        try:
            size = int(self.headers.get("Content-Length", "0"))
            if not (1 <= size <= 2000):
                raise ValueError("Report-Groesse ungueltig")
            report = json.loads(self.rfile.read(size))
            state["uploaded"] = report.get("status") == "ok"
            state["upload_error"] = report.get("message") if not state["uploaded"] else None
            upload_confirmed.set()
            self.send_response(204)
            self.send_header("Content-Length", "0")
            self.end_headers()
        except Exception:
            self.send_error(400)
    def do_GET(self):
        if urlsplit(self.path).path != "/video" or not self.authorized():
            self.send_error(403)
            return
        with state_lock:
            if event_started.is_set():
                self.send_error(409, "Video wird bereits gestreamt")
                return
            event_started.set()
        proc = None
        try:
            self.send_response(200)
            self.send_header("Content-Type", "video/x-matroska")
            self.send_header("Transfer-Encoding", "chunked")
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            print("GitHub verbunden: der ganze Film wird jetzt in 8K umgerechnet.", flush=True)
            proc = subprocess.Popen(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=0)
            threading.Thread(target=progress_reader, args=(proc.stderr,), daemon=True).start()
            transferred = 0
            while True:
                block = proc.stdout.read(1024 * 1024)
                if not block:
                    break
                self.wfile.write(("%X\r\n" % len(block)).encode("ascii"))
                self.wfile.write(block)
                self.wfile.write(b"\r\n")
                transferred += len(block)
            code = proc.wait()
            if code:
                raise RuntimeError("FFmpeg beendete sich mit Code " + str(code))
            self.wfile.write(b"0\r\n\r\n")
            self.wfile.flush()
            state["encoded"] = True
            print("8K-Konvertierung vollstaendig gestreamt:",
                  round(transferred / 1024**3, 2), "GiB. Warte auf Drive-Bestaetigung...", flush=True)
        except Exception as exc:
            print("8K-Stream fehlgeschlagen:", str(exc)[:400], flush=True)
            self.close_connection = True
        finally:
            if proc is not None and proc.poll() is None:
                proc.kill()
                proc.wait()
            encode_finished.set()
            self.close_connection = True

video_server = ThreadingHTTPServer(("127.0.0.1", VIDEO_PORT), VideoOutput)
threading.Thread(target=video_server.serve_forever, daemon=True).start()

# Ein kurzlebiger Tunnel macht nur den Video-Port fuer GitHub erreichbar.
# Keine GitHub-Releases, -Artefakte oder -Videodateien.
cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    response = requests.get(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        timeout=180
    )
    response.raise_for_status()
    cloudflared.write_bytes(response.content)
    cloudflared.chmod(0o755)
log_path = Path("/content/colab_8k_tunnel.log")
log_file = log_path.open("w")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:" + str(VIDEO_PORT)],
    stdout=log_file, stderr=subprocess.STDOUT
)
try:
    tunnel_url = None
    for _ in range(90):
        time.sleep(1)
        log_file.flush()
        contents = log_path.read_text(errors="replace")
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", contents)
        if found:
            tunnel_url = found.group(0)
            break
        if tunnel.poll() is not None:
            raise RuntimeError("Cloudflare-Tunnel startete nicht: " + contents[-1200:])
    if not tunnel_url:
        raise RuntimeError("Cloudflare-Tunnel-URL nicht gefunden: " + contents[-1000:])
    print("Tunnel steht. Starte GitHub-Worker (ohne Video-Upload zu GitHub)...", flush=True)
    dispatch = requests.post(
        "https://api.github.com/repos/" + REPO + "/dispatches",
        headers=headers_gh,
        json={"event_type": "colab-8k-stream", "client_payload": {
            "stream_url": tunnel_url + "/video",
            "stream_key": stream_key,
            "target_name": OUTPUT_NAME,
        }},
        timeout=45,
    )
    if dispatch.status_code != 204:
        raise RuntimeError("GitHub konnte Worker nicht starten (HTTP %s): %s" %
                           (dispatch.status_code, dispatch.text[:600]))
    print("GitHub Actions: https://github.com/" + REPO + "/actions/workflows/colab-8k-direct.yml")
    print("Warte auf GitHub-Verbindung. Colab bitte offen lassen.", flush=True)
    connection_deadline = time.monotonic() + 1800
    while not event_started.wait(timeout=10):
        if upload_confirmed.is_set():
            raise RuntimeError("GitHub konnte den Upload nicht starten: " + str(state["upload_error"]))
        if time.monotonic() > connection_deadline:
            raise RuntimeError("GitHub hat den Stream innerhalb von 30 Minuten nicht gestartet.")
    if not encode_finished.wait(timeout=6 * 3600):
        raise RuntimeError("8K-Stream hat die 6-Stunden-Grenze erreicht.")
    if not state["encoded"]:
        raise RuntimeError("8K-Konvertierung/Streaming nicht erfolgreich.")
    if upload_confirmed.wait(timeout=1800) and state["uploaded"]:
        print("ERFOLG! Google Drive -> Video Converter/Fertig/" + OUTPUT_NAME, flush=True)
    elif upload_confirmed.is_set():
        raise RuntimeError("GitHub-Upload fehlgeschlagen: " + str(state["upload_error"]))
    else:
        print("Film gestreamt, aber noch keine GitHub-Bestaetigung. Status bei GitHub Actions pruefen.")
finally:
    tunnel.terminate()
    try:
        tunnel.wait(timeout=10)
    except subprocess.TimeoutExpired:
        tunnel.kill()
    log_file.close()
    video_server.shutdown()
    input_server.shutdown()
